In [ ]:
import os

input_folder = r"C:\Users\dbecerraq\Downloads\img_sample_input\img_sample_input\data"

def get_png_images(folder):
    return [f for f in os.listdir(folder) if f.lower().endswith(".png")]


images = get_png_images(input_folder)

print (f"se encontro {len(images)} imagenes png:")
for img in images:
    print(img)

se encontro 48 imagenes png:
SEN_3FA7-C478-8182-5D66-3A09-0B62-646B-D288-6476-F3BD-D168-C222-1CE6-D5DE-84EA-387C_1771686087557_pag1.png
SEN_3FA7-C478-8182-5D66-3A09-0B62-646B-D288-6476-F3BD-D168-C222-1CE6-D5DE-84EA-387C_1771686087557_pag2.png
SEN_3FA7-C478-8182-5D66-3A09-0B62-646B-D288-6476-F3BD-D168-C222-1CE6-D5DE-84EA-387C_1771686088934_pag1.png
SEN_3FA7-C478-8182-5D66-3A09-0B62-646B-D288-6476-F3BD-D168-C222-1CE6-D5DE-84EA-387C_1771686088934_pag2.png
SEN_3FA7-C478-8182-5D66-3A09-0B62-646B-D288-6476-F3BD-D168-C222-1CE6-D5DE-84EA-387C_1771686090192_pag1.png
SEN_3FA7-C478-8182-5D66-3A09-0B62-646B-D288-6476-F3BD-D168-C222-1CE6-D5DE-84EA-387C_1771686090192_pag2.png
SEN_3FA7-C478-8182-5D66-3A09-0B62-646B-D288-6476-F3BD-D168-C222-1CE6-D5DE-84EA-387C_1771686091465_pag1.png
SEN_3FA7-C478-8182-5D66-3A09-0B62-646B-D288-6476-F3BD-D168-C222-1CE6-D5DE-84EA-387C_1771686091465_pag2.png
SEN_3FA7-C478-8182-5D66-3A09-0B62-646B-D288-6476-F3BD-D168-C222-1CE6-D5DE-84EA-387C_1771686092729_pag1.png
SEN_3FA7

In [1]:
import os
import cv2
import numpy as np


def crear_mosaico_batch(paths, size=(256, 256), margin=10):
    imgs = []
    
    for path in paths:
        img = cv2.imread(path)
        if img is None:
            continue
        
        img = cv2.resize(img, size)
        imgs.append(img)
    
    # 🔹 Rellenar si faltan (ahora 6 en vez de 4)
    while len(imgs) < 6:
        imgs.append(np.zeros((size[1], size[0], 3), dtype=np.uint8))
    
    h, w = size[1], size[0]

    # 🔹 Tamaño total del lienzo (2x3 con márgenes)
    canvas_h = 2*h + 3*margin
    canvas_w = 3*w + 4*margin

    mosaico = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)

    # 🔹 Posiciones (2 filas x 3 columnas)
    posiciones = [
        (margin, margin),
        (margin, 2*margin + w),
        (margin, 3*margin + 2*w),

        (2*margin + h, margin),
        (2*margin + h, 2*margin + w),
        (2*margin + h, 3*margin + 2*w)
    ]

    for img, (y, x) in zip(imgs, posiciones):
        mosaico[y:y+h, x:x+w] = img

    return mosaico


carpeta = r"C:\Users\dbecerraq\Downloads\img_sample_input\img_sample_input\data"

path_folder_to_save_mosaico = r"C:\Users\dbecerraq\Downloads\img_sample_input\img_sample_input\results"

# Solo PNG
archivos = sorted([
    f for f in os.listdir(carpeta)
    if f.lower().endswith(".png")
])

# 🔹 ahora en bloques de 6
for i in range(0, len(archivos), 6):
    batch = archivos[i:i+6]
    paths = [os.path.join(carpeta, f) for f in batch]
    
    mosaico = crear_mosaico_batch(paths, size=(640,640), margin=15)
    
    path_to_save_mosaico = os.path.join(
        path_folder_to_save_mosaico,
        f"mosaico_{i//6}.png"
    )
    
    cv2.imwrite(path_to_save_mosaico, mosaico)

In [19]:
import os 
import cv2
import numpy as np
import math


# modificador de diametros
imagenes_por_mosaico = 6
cols = 3
rows = 2  # None es igaual automático


def crear_mosaico_batch(paths, size=(256, 256), margin=10, n_imgs=6, cols=3, rows=None):
    imgs = []
    
    for path in paths:
        img = cv2.imread(path)
        if img is None:
            continue
        
        img = cv2.resize(img, size)
        imgs.append(img)
    
    # rellenar si faltan imágenes
    while len(imgs) < n_imgs:
        imgs.append(np.zeros((size[1], size[0], 3), dtype=np.uint8))
    
    h, w = size[1], size[0]

    # filas dinamicas
    if rows is None:
        rows = math.ceil(n_imgs / cols)

    # tamaño del lienzo
    canvas_h = rows * h + (rows + 1) * margin
    canvas_w = cols * w + (cols + 1) * margin

    mosaico = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)

    # colocar imagenes
    for i in range(n_imgs):
        fila = i // cols
        col = i % cols

        y = margin + fila * (h + margin)
        x = margin + col * (w + margin)

        mosaico[y:y+h, x:x+w] = imgs[i]

    return mosaico

# rutas
carpeta = r"C:\Users\dbecerraq\Downloads\img_sample_input\img_sample_input\data"
output = r"C:\Users\dbecerraq\Downloads\img_sample_input\img_sample_input\results"

#lctura de archivos
archivos = sorted([
    f for f in os.listdir(carpeta)
    if f.lower().endswith(".png")
])


# proseso en lotes
for i in range(0, len(archivos), imagenes_por_mosaico):
    batch = archivos[i:i+imagenes_por_mosaico]
    paths = [os.path.join(carpeta, f) for f in batch]
    
    mosaico = crear_mosaico_batch(
        paths,
        size=(640, 640),
        margin=15,
        n_imgs=imagenes_por_mosaico,
        cols=cols,
        rows=rows
    )
    
    cv2.imwrite(
        os.path.join(output, f"mosaico_{i//imagenes_por_mosaico}.png"),
        mosaico
    )